# Troubleshooting missing solutions

:::{autolink-concat}
:::

When {func}`.generate_transitions` cannot find any transition, it raises a {class}`RuntimeError` that carries an {class}`.ExecutionInfo` object. This object lists two kinds of conservation rules for each node and edge:

- **Violated** rules were checked and failed for every combination of quantum numbers. They explain why the reaction is forbidden.
- **Not executed** rules were never checked. The solver works by backtracking: once a rule fails for every candidate value, the solver abandons that branch, so rules that come later are never called. A rule is also not executed if a quantum number that it requires is not available. Not executed rules are therefore _not_ necessarily violated.

The error is a {class}`.NoSolutionsError`, which gives access to the {class}`.ExecutionInfo` through its {attr}`~.NoSolutionsError.execution_info` attribute. The same rules are also listed in warnings, which are switched off on this page to keep the output short.

In [ ]:
import logging
import warnings

from IPython.display import display

import qrules
from qrules.transition import NoSolutionsError

warnings.filterwarnings("ignore", category=RuntimeWarning)
logging.getLogger("qrules").setLevel(logging.ERROR)

## One violated rule, the rest not executed

The decay $J/\psi \to \pi^+\pi^-$ is forbidden in the strong interaction, because the $J/\psi$ has $G=-1$, while a $\pi^+\pi^-$ pair with $I=0$ has $G=+1$.

In [ ]:
try:
    qrules.generate_transitions(
        "J/psi(1S)", ["pi+", "pi-"], allowed_interaction_types="strong"
    )
except NoSolutionsError as error:
    display(error.execution_info)

Only {func}`.g_parity_conservation` is violated. All other rules are listed as not executed, even though charge, baryon number, etc. are conserved in this decay. The solver checks G-parity early on, finds that it fails for every candidate, and stops before it reaches the other rules.

## Several violated rules

In $\eta \to \pi^+\pi^-\pi^0$, the solver checks every rule before giving up, so no rule remains unexecuted. G-parity is violated on both nodes: the $\eta$ has $G=+1$ and three pions have $G=-1$. Mass conservation is violated on the first node, because the $\eta$ is not heavy enough to produce any of the intermediate states that would conserve the other quantum numbers.

In [ ]:
try:
    qrules.generate_transitions(
        "eta", ["pi+", "pi-", "pi0"], allowed_interaction_types="strong"
    )
except NoSolutionsError as error:
    display(error.execution_info)

## No violated rule in common

A reaction is split into many problem sets: one for each topology and each combination of interaction settings, such as the orbital angular momentum $L$ and coupled spin $S$. A rule is only reported as violated if it is violated in **all** of these problem sets. If each problem set fails on a different rule, the list of violated rules ends up empty.

This happens for $J/\psi \to K^0 \Sigma^+ \bar{p}$ via a $\Delta(1910)^+$ resonance, with only strong interactions allowed:

In [ ]:
try:
    qrules.generate_transitions(
        initial_state="J/psi(1S)",
        final_state=["K0", "Sigma+", "p~"],
        allowed_intermediate_particles=["Delta(1910)+"],
        allowed_interaction_types="strong",
        formalism="canonical-helicity",
    )
except NoSolutionsError as error:
    display(error.execution_info)

In such cases, relax the problem to find the cause. The $J/\psi$ has isospin $I=0$, while $\Delta^+\bar{p}$ can only couple to $I=1$ or $I=2$, so the first decay violates isospin conservation in the strong interaction. Electromagnetic interactions do not conserve isospin, so allowing them results in solutions:

In [ ]:
reaction = qrules.generate_transitions(
    initial_state="J/psi(1S)",
    final_state=["K0", "Sigma+", "p~"],
    allowed_intermediate_particles=["Delta(1910)+"],
    allowed_interaction_types=["strong", "EM"],
    formalism="canonical-helicity",
)
len(reaction.transitions)

Use {meth}`.StateTransitionManager.set_allowed_interaction_types` (see {ref}`usage/reaction:Select interaction types`) or remove rules from the {class}`.NodeSettings` to narrow down which rule excludes a decay. For a quick check of a reaction without intermediate states, use {func}`.check_reaction_violations` (see {doc}`/usage`). See {doc}`/usage/conservation` for how individual rules can be checked by hand.